# 02 — Retrieval Engines Sanity Check

Builds all three retrieval engines (BM25, Vector, Hybrid) and verifies they
work correctly by running a sanity-check query side-by-side.

**Prerequisites:** Run `01_data_collection.ipynb` first to generate `../data/chunks.json`.

**Purpose:** Validate that all three engines return results in the expected schema
before running the full experiment in `03_experiment.ipynb`.

In [ ]:
# Cell 1 — Install dependencies and imports
!pip install rank-bm25 sentence-transformers faiss-cpu

import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.chunker import load_chunks
from src.bm25_retriever import BM25Retriever
from src.vector_retriever import VectorRetriever
from src.hybrid_retriever import HybridRetriever

In [ ]:
# Cell 2 — Load data
chunks = load_chunks("../data/chunks.json")
print(f"Loaded {len(chunks)} chunks")

In [ ]:
# Cell 3 — Build all indexes
bm25 = BM25Retriever()
bm25.build_index(chunks)

vector = VectorRetriever()
vector.build_index(chunks)

hybrid = HybridRetriever(bm25, vector)

In [ ]:
# Cell 4 — Run sanity-check query and display side-by-side
TEST_QUERY = "Who is the best player in the 2026 FIFA World Cup?"
TOP_K = 3

bm25_results   = bm25.search(TEST_QUERY, top_k=TOP_K)
vector_results = vector.search(TEST_QUERY, top_k=TOP_K)
hybrid_results = hybrid.search(TEST_QUERY, top_k=TOP_K)

print(f"Query: '{TEST_QUERY}'")
print()

# Print side-by-side table
col_w = 42  # characters per column
header = f"{'Rank':<5} | {'BM25':<{col_w}} | {'Vector':<{col_w}} | {'Hybrid':<{col_w}}"
sep    = "-" * len(header)
print(header)
print(sep)

for i in range(TOP_K):
    def fmt(results, idx):
        if idx >= len(results):
            return "(none)"
        r = results[idx]
        cell = f"[{r['source'][:18]}] {r['text'][:20]}..."
        return cell[:col_w]

    print(f"{i+1:<5} | {fmt(bm25_results, i):<{col_w}} | {fmt(vector_results, i):<{col_w}} | {fmt(hybrid_results, i):<{col_w}}")

print()
print("Full top-1 results:")
print(f"  BM25   score={bm25_results[0]['score']:.4f}  | {bm25_results[0]['source']}")
print(f"    {bm25_results[0]['text'][:120]} ...")
print(f"  Vector score={vector_results[0]['score']:.4f}  | {vector_results[0]['source']}")
print(f"    {vector_results[0]['text'][:120]} ...")
print(f"  Hybrid rrf={hybrid_results[0]['rrf_score']:.5f} | bm25_rank={hybrid_results[0]['bm25_rank']} | vec_rank={hybrid_results[0]['vector_rank']} | {hybrid_results[0]['source']}")
print(f"    {hybrid_results[0]['text'][:120]} ...")

In [ ]:
# Cell 5 — Verify output schema
BM25_REQUIRED_KEYS   = {'rank', 'score', 'chunk_id', 'source', 'text'}
VECTOR_REQUIRED_KEYS = {'rank', 'score', 'chunk_id', 'source', 'text'}
HYBRID_REQUIRED_KEYS = {'rank', 'rrf_score', 'bm25_rank', 'vector_rank', 'chunk_id', 'source', 'text'}

for i, result in enumerate(bm25_results):
    missing = BM25_REQUIRED_KEYS - set(result.keys())
    assert not missing, f"BM25 result {i} missing keys: {missing}"

for i, result in enumerate(vector_results):
    missing = VECTOR_REQUIRED_KEYS - set(result.keys())
    assert not missing, f"Vector result {i} missing keys: {missing}"

for i, result in enumerate(hybrid_results):
    missing = HYBRID_REQUIRED_KEYS - set(result.keys())
    assert not missing, f"Hybrid result {i} missing keys: {missing}"

# Verify rank is 1-indexed
assert bm25_results[0]['rank'] == 1,   "BM25 rank should start at 1"
assert vector_results[0]['rank'] == 1, "Vector rank should start at 1"
assert hybrid_results[0]['rank'] == 1, "Hybrid rank should start at 1"

print("All schema checks passed")